In [6]:
from contextlib import contextmanager
import lseg.data as ld
# import matplotlib.pyplot as plt
# import matplotlib.dates as mdates
from lseg.data.discovery import Chain
from IPython.display import display, clear_output
# import seaborn as sns
from datetime import date
import pandas as pd
from functools import reduce

# end_date = (datetime.now() - timedelta(weeks=1)).strftime('%Y-%m-%d')
pd.set_option('future.no_silent_downcasting', True)

@contextmanager
def ld_session():
    """Context manager for handling a LSEG Data Library for Python session."""
    try:
        # Create a session and return it
        ld.open_session()
        yield
    finally:
        # Close the session
        ld.close_session()
        
today_str = date.today().strftime('%Y-%m-%d')

In [2]:
# Initialize dictionary
mena_banks_by_country = {}

with ld.open_session():
    # 1. Get the top-level constituents (the country-specific chains)
    main_chain = Chain(name="0#.TRXFLDMEPUBANK")
    country_chains = main_chain.constituents
    
    # 2. Loop through and pull constituents for each country chain separately
    for tic in country_chains[1:]:
        # Create a dictionary entry for each country's sub-chain
        sub_chain = Chain(name=tic)
        mena_banks_by_country[tic] = sub_chain.constituents

In [7]:
all_country_dfs = []

with ld.open_session():
    for country_ric, tickers in mena_banks_by_country.items():
        if not tickers:
            continue
            
        print(f"Pulling data for {country_ric}...")
        
        # Pulling country-specific data
        df = ld.get_history(
            universe=tickers, 
            fields=['TR.F.TotAssets', 'TR.F.ComEqTot', 'TR.F.TotLiab', 'TR.F.TotLiabEq'],
            parameters={
                'Curn': 'Native', 
                'Period': 'FQ0', 
                'Frq': 'FQ',      # Quarterly frequency for fundamentals
                'SDate': '0', 
                'EDate': '2010-01-01'   
            }
        )
        
        # Optional: Add a column to identify the source country chain
        df['Country_Chain'] = country_ric
        all_country_dfs.append(df)

# To get the intersection of all countries (only dates where all have reported)
intersected_fundamentals = reduce(lambda left, right: left.join(right, how='inner', rsuffix='_drop'), all_country_dfs)
display(intersected_fundamentals.head())

Pulling data for 0#.TRXFLDAEPBANK...
Pulling data for 0#.TRXFLDBHPBANK...
Pulling data for 0#.TRXFLDEGPBANK...
Pulling data for 0#.TRXFLDQAPBANK...
Pulling data for 0#.TRXFLDMAPBANK...
Pulling data for 0#.TRXFLDOMPBANK...
Pulling data for 0#.TRXFLDKWPBANK...


AJBNK.DU                                          \
           Total Assets Common Equity - Total Total Liabilities   
Date                                                              
2009-12-31         <NA>                  <NA>              <NA>   
2010-03-31   2169968000             989143000        1180825000   
2010-06-30   2595447000             987101000        1608346000   
2010-09-30   2923798000             983821000        1939977000   
2010-12-31   3234346000            1006942000        2227404000   

                                           DISB.DU                        \
           Total Liabilities & Equity Total Assets Common Equity - Total   
Date                                                                       
2009-12-31                       <NA>  84304271000            8975890000   
2010-03-31                 2169968000  84971466000            8712483000   
2010-06-30                 2595447000  83803879000            8857855000   
2010-09-30                 2923798000  81847502000            9223282000   
2010-12-31                 3234346000  89884401000            9326079000   

                                                              ENBD.DU  \
           Total Liabilities Total Liabilities & Equity  Total Assets   
Date                                                                    
2009-12-31       75323471000                84304271000  281576482000   
2010-03-31       76258160000                84971466000  289723434000   
2010-06-30       74944285000                83803879000  282340152000   
2010-09-30       72621737000                81847502000  284222120000   
2010-12-31       79611314000                89884401000  286078324000   

                                  ...                    KIBK.KW      KPRO.KW  \
           Common Equity - Total  ... Total Liabilities & Equity Total Assets   
Date                              ...                                           
2009-12-31           27876499000  ...                 1140336000   5326765000   
2010-03-31           29539636000  ...                 1100628000   5377073000   
2010-06-30           28686491000  ...                 1149864000   5292361000   
2010-09-30           28667097000  ...                 1135005000   5335766000   
2010-12-31           29655770000  ...                 1141860000   5661163000   

                                                                               \
           Common Equity - Total Total Liabilities Total Liabilities & Equity   
Date                                                                            
2009-12-31             544327000        4458784000                 5326765000   
2010-03-31             527285000        4498890000                 5377073000   
2010-06-30             533687000        4351715000                 5292361000   
2010-09-30             543554000        4420027000                 5335766000   
2010-12-31             559940000        4649173000                 5661163000   

                GBKK.KW                                          \
           Total Assets Common Equity - Total Total Liabilities   
Date                                                              
2009-12-31   4743911000             407853000        4336058000   
2010-03-31   4515779000             397352000        4118427000   
2010-06-30   4513295000             391228000        4122067000   
2010-09-30   4599473000             397716000        4201757000   
2010-12-31   4599777000             410715000        4189062000   

                                      Country_Chain_drop  
           Total Liabilities & Equity                     
Date                                                      
2009-12-31                 4743911000   0#.TRXFLDKWPBANK  
2010-03-31                 4515779000   0#.TRXFLDKWPBANK  
2010-06-30                 4513295000   0#.TRXFLDKWPBANK  
2010-09-30                 4599473000   0#.TRXFLDKWPBANK  
2010-12-31                 459977700

In [ ]:
# These are Banking-sector benchmarks
# This creates a new list with '0#' gone
benchmarksRic = list(map(lambda ric: ric.replace('0#', ''), country_chains))

with ld.open_session():
    benchm = ld.get_history(
                universe=benchmarksRic, 
                fields='TR.PriceClose', # TR.PriceClose field always returns adjusted stock prices.
                parameters={'Curn': 'Native', 'Fill': 'None'},
                start='2009-12-31', 
                end=today_str, 
                interval='daily')
    benchInf = ld.get_data(universe=benchmarksRic, fields=["TR.IndexGeography","TR.IndexSeriesName"])
    

display(benchm.head())
display(benchInf)

Price Close,.TRXFLDMEPUBANK,.TRXFLDAEPBANK,.TRXFLDBHPBANK,.TRXFLDEGPBANK,.TRXFLDQAPBANK,.TRXFLDMAPBANK,.TRXFLDOMPBANK,.TRXFLDKWPBANK
Date,,,,,,,,
2025-12-31,432.83,885.28,164.92,5794.34,379.45,491.03,178.03,388.54
2026-01-01,433.99,885.28,164.92,5794.34,379.45,491.03,178.61,388.54
2026-01-02,435.07,893.32,<NA>,<NA>,<NA>,497.93,<NA>,<NA>
2026-01-05,433.41,892.16,165.17,5656.05,387.83,499.94,181.83,385.57
2026-01-06,432.25,898.77,165.37,5758.69,389.09,502.58,182.76,385.48


,Instrument,Index Geography,Index Series Name
0,.TRXFLDMEPUBANK,Middle East,TR Indices EMA Series
1,.TRXFLDAEPBANK,United Arab Emirates,TR Indices EMA Series
2,.TRXFLDBHPBANK,Bahrain,TR Indices EMA Series
3,.TRXFLDEGPBANK,Egypt,TR Indices EMA Series
4,.TRXFLDQAPBANK,Qatar,TR Indices EMA Series
5,.TRXFLDMAPBANK,Morocco,TR Indices EMA Series
6,.TRXFLDOMPBANK,Oman,TR Indices EMA Series
7,.TRXFLDKWPBANK,Kuwait,TR Indices EMA Series


In [ ]:
# here is TR Indices EMA Series (GCC PRice return index) and S&P GCC COMPOSITE (preferred?)
benchmarkMainRic = ['.TRXFLDGCPU', '.GPDGC']

with ld.open_session():
    benchmMain = ld.get_history(
                universe=benchmarkMainRic, 
                fields='TR.PriceClose',
                parameters={'Curn': 'Native', 'Fill': 'None'},
                start='2009-12-31', 
                end=today_str, 
                interval='daily')
    benchMainInf = ld.get_data(universe=benchmarkMainRic, fields=["TR.IndexGeography","TR.IndexSeriesName"])

display(benchmMain.head())
display(benchMainInf)

.TRXFLDGCPU,Price Close
Date,
2010-09-16,157.02658
2010-09-17,157.02658
2010-09-20,158.06
2010-09-21,158.45
2010-09-22,158.46


,Instrument,Index Geography,Index Series Name
0,.TRXFLDGCPU,Middle East,TR Indices EMA Series


In [ ]:
country_data_frames = {}
fields_to_pull = ['TR.TotalReturn', 'TR.PriceClose', 'TR.CompanyMarketCapitalization']

with ld.open_session():
    for country_ric, tickers in mena_banks_by_country.items():
        if not tickers: continue
        
        field_frames = []
        for field in fields_to_pull:
            df_field = ld.get_history(
                universe=tickers, 
                fields=[field],
                parameters={'Curn': 'Native', 'Fill': 'None'},
                start='2009-12-31', 
                end=today_str, 
                interval='daily'
            )
            
            if not df_field.empty:
                # 1. Clean duplicates for this specific field
                df_field = df_field.groupby(df_field.index).first()
                
                # 2. FORCE MULTIINDEX: Create (RIC, Field) tuples for columns
                # This ensures every column is explicitly labeled
                new_columns = pd.MultiIndex.from_product([df_field.columns, [field]], names=['RIC', 'Field'])
                df_field.columns = new_columns
                
                field_frames.append(df_field)
        
        if field_frames:
            # Join fields for this country
            country_df = pd.concat(field_frames, axis=1)
            country_data_frames[country_ric] = country_df

# Final Intersection (Inner Join)
intersected_eq_data = pd.concat(country_data_frames.values(), axis=1, join='inner')

# NOW this will work beautifully:
display(intersected_eq_data['ENBD.DU'].head())

Field,TR.TotalReturn,TR.PriceClose,TR.CompanyMarketCapitalization
Date,,,
2025-12-31,-1.590106,27.85,175917261346.049988
2026-01-05,2.097902,29.2,184444668987.600006
2026-01-06,0.171233,29.25,184760498900.25
2026-01-08,0.170068,29.45,186023818550.850006
2026-01-12,1.362862,29.75,187918798026.75
